In [1]:
!pip install ultralytics torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00a 0:00:01


In [2]:
# 1. Install necessary libraries
!pip install -q ultralytics torchvision pandas

import os
import yaml
import glob
from IPython.display import Image, display

# 2. Dynamically find the dataset path in Kaggle
# Kaggle mounts data in /kaggle/input/. We look for the folder containing 'train' and 'valid'
base_path = "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset"
dataset_path = None

for root, dirs, files in os.walk(base_path):
    if 'train' in dirs and 'valid' in dirs:
        dataset_path = root
        break

if dataset_path is None:
    raise FileNotFoundError("Could not find the dataset! Did you remember to click 'Add Data'?")

print(f"✅ Dataset found at: {dataset_path}")

# 3. Create the YAML configuration for the Train-Validation split
pku_classes = ["Missing_hole", "Mouse_bite", "Open_circuit", "Short_circuit", "Spur", "Spurious_copper"]

yaml_content = {
    "path": dataset_path,
    "train": "train/images",  
    "val": "valid/images",    
    "nc": 6,
    "names": pku_classes
}

# Save config to Kaggle's working directory
YAML_PATH = "/kaggle/working/pku_finetune_config.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ Data Preprocessing & Split Configured.")
# Note: Image resizing and normalization (Task 1) are natively handled by YOLO in the next step.

✅ Dataset found at: /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)
✅ Data Preprocessing & Split Configured.


In [ ]:
from ultralytics import YOLO
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# --- 1. YOLOv11 Implementation ---
print("🚀 Initializing YOLOv11 for Training...")
model_yolo = YOLO("yolo11s.pt") # Loads the small YOLOv11 base model

# Train with Hyperparameter Tuning
results = model_yolo.train(
    data=YAML_PATH,
    epochs=10,         # CHANGED: Reduced to 10 epochs for a very fast training run
    batch=32,          # Keeps the GPU fully utilized
    imgsz=416,         # Smaller image size for faster processing
    device=0,          
    freeze=10,         
    degrees=15.0,      
    fliplr=0.5,        
    cache=True,        # Keeps data in RAM to prevent disk bottlenecks
    workers=8,         
    amp=True,          
    patience=10,       
    project="/kaggle/working/transfer-learning",
    name="pku_yolo_model_fast"
)

# --- 2. CNN / ResNet Comparison ---
print("🔄 Initializing ResNet-50 Faster R-CNN for architectural comparison...")
model_resnet = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
num_classes = 7 # 6 defect classes + 1 background

# Replace classification head
in_features = model_resnet.roi_heads.box_predictor.cls_score.in_features
model_resnet.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
print("✅ ResNet-50 initialized. (Note: YOLO is selected for final deployment due to real-time speed advantages).")

In [ ]:
# 1. Evaluate using IoU and mAP
print("📊 Evaluating Object Detection Metrics...")
metrics = model_yolo.val(
    data=YAML_PATH, 
    split='val', 
    batch=8, 
    iou=0.5 # Strict IoU threshold
)

# 2. Detect and localize defects on sample validation images
VAL_IMAGES_PATH = os.path.join(dataset_path, "valid/images")

print("🔍 Running predictions to draw bounding boxes...")
predict_results = model_yolo.predict(
    source=VAL_IMAGES_PATH,
    conf=0.25, # Confidence threshold
    save=True, # Saves images with drawn bounding boxes
    project="/kaggle/working/final-results",
    name="visual_check"
)

# Display one of the processed images to verify
sample_images = glob.glob('/kaggle/working/final-results/visual_check/*.jpg')
if sample_images:
    print("✅ Sample Detection Output:")
    display(Image(filename=sample_images[0]))

In [ ]:
import pandas as pd

clases = metrics.names
precisiones = metrics.box.p
recalls = metrics.box.r
map50s = metrics.box.ap50

# Calculate F1-Score (Used as a proxy for Accuracy in Object Detection)
f1_scores = 2 * (precisiones * recalls) / (precisiones + recalls + 1e-16)

filas = []
for i, c_idx in enumerate(metrics.box.ap_class_index):
    filas.append({
        "Defect Class": clases[c_idx],
        "Precision": round(precisiones[i], 4),
        "Recall": round(recalls[i], 4),
        "F1-score (Accuracy)": round(f1_scores[i], 4),
        "mAP@50": round(map50s[i], 4)
    })

df = pd.DataFrame(filas)
print("--- FINAL PERFORMANCE METRICS ---")
display(df)

# Evaluate Inference Time 
preprocess = metrics.speed['preprocess']
inference = metrics.speed['inference']
postprocess = metrics.speed['postprocess']
total_time = preprocess + inference + postprocess

print(f"\n--- INFERENCE TIME EVALUATION (Kaggle GPU) ---")
print(f"Pre-process: {preprocess:.2f} ms")
print(f"Inference: {inference:.2f} ms")
print(f"Post-process: {postprocess:.2f} ms")
print(f"Total time per image: {total_time:.2f} ms")
print(f"Estimated FPS (Frames Per Second): {1000 / total_time:.2f}")

In [ ]:
import os
import glob
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

# 0. DEFINIR EL MODELO (¡Faltaba en tu código!)
# Make sure this matches the path where YOLO saved your best weights earlier
NEW_MODEL_PATH = "/kaggle/working/transfer-learning/pku_yolo_model_fast/weights/best.pt"

# 1. CREAR EL "ESPEJO" EN TU OUTPUT
BASE_WORK = "/kaggle/working/pku_eval_clean"
IMG_DIR = os.path.join(BASE_WORK, "images", "val")
LBL_DIR = os.path.join(BASE_WORK, "labels", "val")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(LBL_DIR, exist_ok=True)

# 2. RUTAS ORIGINALES DEL DATASET PKU EN KAGGLE 
ORIGINAL_IMAGES = "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/valid/images"
ORIGINAL_LABELS = "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)/valid/labels"

# Dynamic fallback just in case Kaggle changed the input path slightly
if not os.path.exists(ORIGINAL_IMAGES):
    print("⚠️ Ruta estática no encontrada. Buscando ruta dinámicamente...")
    search = glob.glob("/kaggle/input/**/valid/images", recursive=True)
    if search:
        ORIGINAL_IMAGES = search[0]
        ORIGINAL_LABELS = search[0].replace("images", "labels")

# 3. ENLAZAR IMÁGENES Y ETIQUETAS
print("🔄 Estructurando dataset para YOLO...")
for img_path in glob.glob(os.path.join(ORIGINAL_IMAGES, "*.*")):
    sym_img = os.path.join(IMG_DIR, os.path.basename(img_path))
    if not os.path.exists(sym_img): 
        os.symlink(img_path, sym_img)

for txt_path in glob.glob(os.path.join(ORIGINAL_LABELS, "*.txt")):
    sym_txt = os.path.join(LBL_DIR, os.path.basename(txt_path))
    if not os.path.exists(sym_txt): 
        os.symlink(txt_path, sym_txt)

# 4. CREAR EL YAML A PRUEBA DE FALLOS
yaml_content = {
    "path": BASE_WORK,
    "train": "",
    "val": "images/val",  # YOLO buscará de forma nativa en labels/val
    "nc": 6,
    "names": [
        "Missing_hole", 
        "Mouse_bite", 
        "Open_circuit", 
        "Short_circuit", 
        "Spur", 
        "Spurious_copper"
    ]
}
YAML_SAFE_PATH = "/kaggle/working/safe_eval.yaml"
with open(YAML_SAFE_PATH, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)
print(f"📝 YAML creado en: {YAML_SAFE_PATH}")

# 5. LANZAR LA EVALUACIÓN
print("🚀 Evaluando modelo Fine-Tuneado...")
model = YOLO(NEW_MODEL_PATH)
metrics = model.val(data=YAML_SAFE_PATH, split='val', device='0', plots=True)
print("✅ Validación completada. Revisa tu nueva matriz de confusión.")

# 6. MATRIZ DE CONFUSIÓN PERSONALIZADA
# Extraer y Transponer la matriz
raw_matrix = metrics.confusion_matrix.matrix
transposed_matrix = raw_matrix.T

# AÑADIR "BACKGROUND" A LAS ETIQUETAS
class_names = list(metrics.names.values())
if transposed_matrix.shape[0] > len(class_names):
    class_names.append("Background (Fondo)") # Revelamos la clase oculta

# Dibujar la matriz completa (7x7)
plt.figure(figsize=(12, 10))
ax = sns.heatmap(
    transposed_matrix, annot=True, fmt=".0f", cmap="Greens",
    xticklabels=class_names, yticklabels=class_names,
    linewidths=1, linecolor='black'
)

plt.title("Matriz de Confusión Completa (Con Background)\n", fontsize=18, fontweight='bold', loc='left')
plt.xlabel("Actual Class (Ground Truth)", fontsize=14, fontweight='bold', labelpad=15)
plt.ylabel("Predicted Class", fontsize=14, fontweight='bold', labelpad=15)
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.xticks(rotation=45, ha='left')
plt.yticks(rotation=0)
plt.tight_layout()

# --- NUEVO CÓDIGO PARA GUARDAR ---
ruta_guardado = "/kaggle/working/matriz_completa.png"
# Guardar con alta resolución (dpi=300) y ajustando los bordes (bbox_inches='tight')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
print(f"📁 Imagen guardada exitosamente en: {ruta_guardado}")

# Mostrar la imagen en el notebook después de haberla guardado
plt.show()

In [ ]:
import glob
import os
from ultralytics import YOLO
from IPython.display import display, Image

# 1. Load your trained model weights
model_path = "/kaggle/working/transfer-learning/pku_yolo_model_fast/weights/best.pt"
deployed_model = YOLO(model_path)

# 2. Search for the images in the TEST folder
search_paths = [
    "/kaggle/input/*/test/images/01_PCB__1143.jpg",
    "/kaggle/input/*/*/test/images/01_PCB__1082.jpg",
    "/kaggle/input/*/*/*/test/images/01_PCB__1203.jpg"
]

sample_images = []
for path in search_paths:
    sample_images.extend(glob.glob(path))

# FALLBACK: If it can't find those specific 3, grab a random image from the test folder
if len(sample_images) == 0:
    print("Could not find those exact 3 images. Grabbing a random TEST image instead...")
    sample_images = glob.glob("/kaggle/input/**/test/images/*.jpg", recursive=True)

if len(sample_images) > 0:
    # Grab the first image from our list to test
    test_image_path = sample_images[0] 
    print(f"📷 Testing model on image: {test_image_path}")

    # 3. Run Inference on the single image
    results = deployed_model.predict(
        source=test_image_path,
        conf=0.3,       # Confidence threshold 
        save=True,      # Save the image with drawn bounding boxes
        project="/kaggle/working/single-test",
        name="result",
        exist_ok=True
    )

    # 4. Display the resulting image directly in the notebook
    image_filename = os.path.basename(test_image_path)
    output_image_path = f"/kaggle/working/single-test/result/{image_filename}"
    
    print("\n✅ Detection Complete! Here is the result:")
    display(Image(filename=output_image_path))

else:
    print("⚠️ Still could not find any images! Double check that your dataset has a 'test' folder.")

In [ ]:
model_yolo.export(format="engine",
                 project= "/model")